In [0]:
env = 'p'

In [0]:
# Update values for record where CalendarDate = '2024-03-29'
spark.sql(f'''
            UPDATE `{env}-cdh`.`reference`.`ref_calendar`
            SET IsHoliday = 1
                  , HolidayName = 'Catholic Good Friday'
                  , IsBusinessDay = 0
                  , BusinessDayOfMonth = 0
            WHERE CalendarDate = '2024-03-29';
          ''')


# Update values for record where CalendarDate = '2024-03-28'
spark.sql(f'''
            UPDATE `{env}-cdh`.`reference`.`ref_calendar`
            SET NextBusinessDay = '2024-04-01'
            WHERE CalendarDate = '2024-03-28';
          ''')


# Update values for record where CalendarDate > '2024-03-29'
spark.sql(f'''
            UPDATE `{env}-cdh`.`reference`.`ref_calendar`
            SET PreviousBusinessDay = '2024-03-28'
            WHERE CalendarDate between '2024-03-30' and '2024-04-01';
          ''')

spark.sql(f'''
            UPDATE `{env}-cdh`.`reference`.`ref_calendar`
            SET PreviousMonthLastWorkingDate = '2024-03-28'
            WHERE CalendarDate like '2024-04-%';
          ''')


# Update values for record where CalendarDate like '2024-03-%'
spark.sql(f'''
            UPDATE `{env}-cdh`.`reference`.`ref_calendar`
            SET BusinessDaysInMonth = 18, CurrentMonthLastWorkingDate = '2024-03-28'
            WHERE CalendarDate like '2024-03-%';
          ''')

## DBTC-1114

In [0]:
spark.sql('''
DELETE FROM `p-cdh`.derivativearchive.dparty
WHERE BusinessDate IN (SELECT DISTINCT BusinessDate FROM `qa-cdh`.reference.dparty_testthanos );
''')

In [0]:
spark.sql('''
INSERT INTO `p-cdh`.derivativearchive.dparty
SELECT * FROM `qa-cdh`.reference.dparty_testthanos
 ''')

##DBTC-1149

In [0]:
target_dates_df = spark.sql(f"""select CalendarDate from `{env}-cdh`.reference.ref_calendar
                    where isbusinessday = 1 and 
                    CalendarDate between '2024-12-31' and '2025-01-31'
                    order by CalendarDate asc; """)

In [0]:
from pyspark.sql.functions import col, lit, when, coalesce

for row in target_dates_df.select("CalendarDate").collect():
    try:
        # Extract date from row
        run_date = row['CalendarDate']

        # Special calculation of the IsSecuredFlag for EUCLID
        LastBusinessDateOfPreviousQuarter = spark.sql(f'''
            SELECT CASE
                WHEN IsBusinessDay = 1 THEN CalendarDate
                ELSE PreviousBusinessDay
            END AS LastBusinessDateOfPreviousQuarter
            FROM `{env}-cdh`.reference.ref_calendar
            WHERE CalendarDate = date_add('{run_date}', -1 * cast((
                SELECT DayOfQuarter
                FROM `{env}-cdh`.reference.ref_calendar
                WHERE CalendarDate = '{run_date}'
            ) as int))
        ''').first()[0]

        # Get distinct BusinessDate for EUCLID portfolio
        danalyticsmetricsdate = spark.sql(f'''
            SELECT distinct BusinessDate
            FROM `{env}-cdh`.derivative.danalyticsmetrics
            WHERE PortfolioName='EUCLID'
        ''')

        danalyticsmetricsdate.createOrReplaceTempView("danalyticsmetricsdate")

        IsSecuredFlagEuclidMinDate = spark.sql(f'''
            SELECT min(BusinessDate) IsSecuredFlagEuclidMinDate
            FROM danalyticsmetricsdate
            WHERE BusinessDate > '{LastBusinessDateOfPreviousQuarter}'
        ''').first()[0]

        # Filter for business date of last quarter end
        danalyticsmetricsEndOfPreviousQuarter = danalyticsmetricsdate.filter(
            danalyticsmetricsdate.BusinessDate == LastBusinessDateOfPreviousQuarter
        )

        def get_secured_agreements_query(date):
            return spark.sql(f'''
                SELECT distinct AgreementKey
                FROM `{env}-cdh`.derivative.danalyticsmetrics
                WHERE businessdate = '{date}'
                AND AgreementKey IN (
                    SELECT distinct AgreementKey
                    FROM `{env}-cdh`.primary.collateral 
                    WHERE sourcesystem = 'thesis' 
                    AND collateralstatus = 'Ενεργή'
                    AND businessdate = '{date}'
                )
                GROUP BY AgreementKey
                HAVING SUM(CREVBaseTLC) > 20000
            ''')

        # Get secured agreements based on business date
        if not danalyticsmetricsEndOfPreviousQuarter.isEmpty():
            SecuredAgreements = get_secured_agreements_query(LastBusinessDateOfPreviousQuarter)
        else:    
            SecuredAgreements = get_secured_agreements_query(IsSecuredFlagEuclidMinDate)

        SecuredAgreements.createOrReplaceTempView("SecuredAgreements")

        IsSecuredFlagEuclid = spark.sql(f'''
            SELECT distinct AgreementKey,
            CASE
                WHEN AgreementKey IN (SELECT AgreementKey FROM SecuredAgreements) THEN True
                ELSE False
            END AS is_secured_flag
            FROM `{env}-cdh`.primary.Exposure
            WHERE BusinessDate = '{run_date}' AND PortfolioName='EUCLID'
        ''')

        IsSecuredFlagEuclid = IsSecuredFlagEuclid.filter(IsSecuredFlagEuclid.is_secured_flag == True).distinct()

        # Collateral thesis secured data
        collateral_thesis_secured = spark.sql(f"""
            SELECT
                AgreementKey,
                CategoryCode,
                CollateralStatusCode,
                RealEstateCollateralStatusCode,
                SourceSystem,
                PortfolioName,
                BusinessDate
            FROM `{env}-cdh`.primary.Collateral
            WHERE BusinessDate = '{run_date}'
            and SourceSystem = 'thesis'
        """)

        collateral_thesis_secured = collateral_thesis_secured.withColumnRenamed(
            "RealEstateCollateralStatusCode", "RealEstateCollateralStatusCode_secured"
        ).withColumnRenamed(
            "AgreementKey", "AgreementKey_secured"
        )

        collateral_thesis_secured = collateral_thesis_secured.filter(
            (collateral_thesis_secured.RealEstateCollateralStatusCode_secured == 'Ενεργή')
            & (collateral_thesis_secured.PortfolioName != 'EUCLID')
        )

        secured_flag_keys = collateral_thesis_secured.withColumn(
            'is_secured_flag', lit(True)
        ).select("AgreementKey_secured", "is_secured_flag").distinct()

        secured_flag_keys = secured_flag_keys.union(IsSecuredFlagEuclid)

        # Exposure profits query
        exposure_profits = spark.sql(f'''
            SELECT
                AgreementKey AS AgreementKeyProfits,
                PrimaryPartyFlag,
                ExposureKey,
                BusinessDate
            FROM `{env}-cdh`.primary.exposure
            WHERE BusinessDate = '{run_date}' 
            and SourceSystem = 'profits'
            and PrimaryPartyFlag == True
        ''')

        exposure_profits_collateral_secured_join_condition = (
            exposure_profits["AgreementKeyProfits"] == secured_flag_keys["AgreementKey_secured"]
        )

        joined_df = exposure_profits.join(
            secured_flag_keys,
            exposure_profits_collateral_secured_join_condition,
            "left"
        )

        joined_df = joined_df.withColumn(
            "IsSecuredFlagCalc",
            coalesce(secured_flag_keys.is_secured_flag, lit(False))
        ).select(
            "AgreementKeyProfits", "IsSecuredFlagCalc", "BusinessDate"
        ).distinct()
        joined_df.createOrReplaceTempView("joined_df")

        spark.sql(f"""
            MERGE INTO `{env}-cdh`.derivativeactive.DExposure dexp
            USING joined_df 
            ON joined_df.AgreementKeyProfits = dexp.AgreementKey
            AND joined_df.BusinessDate = dexp.BusinessDate
            AND dexp.SourceSystem = 'profits'
            WHEN MATCHED THEN UPDATE SET dexp.IsSecuredFlag = joined_df.IsSecuredFlagCalc
            """)

        print(f"Date {run_date} completed")

    except Exception as e:
        print(f"Error in date {run_date}: {e}")

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `{env}-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')